## Frontend Live streaming chatbot using FastAPI

##### this will work on .py and react , here just for reference and  study i give this here 

In [1]:
import langchain
import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware

import json


# --------------------------------------------------
# Load environment variables
# --------------------------------------------------

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    api_key=groq_api_key,
    model="openai/gpt-oss-120b",
    temperature=0.2
)


# --------------------------------------------------
# FastAPI app
# --------------------------------------------------

app = FastAPI()


# --------------------------------------------------
# CORS
# --------------------------------------------------

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000"],
    allow_credentials=True,
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["*"],
)


# --------------------------------------------------
# Chat endpoint
# --------------------------------------------------

@app.post("/chat")
async def chat_endpoint(request: Request):

    data = await request.json()

    user_input = data.get("user_input", "")

    if not user_input:
        return {"error": "No user input provided."}


    # --------------------------------------------------
    # Prompt template
    # --------------------------------------------------

    prompt = PromptTemplate.from_template(
        """You are a helpful AI assistant.
Answer the question below shortly and briefly.

Question: {question}"""
    )


    # --------------------------------------------------
    # Chain
    # --------------------------------------------------

    chat_chain = prompt | llm | StrOutputParser()


    # --------------------------------------------------
    # Streaming generator
    # --------------------------------------------------

    def generate():

        try:

            for chunk in chat_chain.stream(
                {"question": user_input}
            ):

                event_data = json.dumps({
                    "answer": chunk
                })

                yield f"data: {event_data}\n\n"


        except Exception as e:

            error_data = json.dumps({
                "error": str(e)
            })

            yield f"data: {error_data}\n\n"


        finally:

            yield "data: [DONE]\n\n"


    # --------------------------------------------------
    # Return streaming response
    # --------------------------------------------------

    return StreamingResponse(
        content=generate(),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "Connection": "keep-alive",
        }
    )

# list keep entire thing in memory but generator function yield one at a time and keep memory usage lowx``
# generater function overview
# def get_all_numbers():
#     return [i for i in range(100000)]  # Creates entire list

# # Generator function - one at a time
# def get_numbers():
#     for i in range(100000):
#         yield i  # Only creates one number at a time


# def countdown():
#     yield 3
#     yield 2
#     yield 1
#     yield "Blast off!"


# # Usage
# gen = countdown()
# print(next(gen))  # 3
# print(next(gen))  # 2
# print(next(gen))  # 1
# print(next(gen))  # "Blast off!"


# Any yield = Generator






React code

In [ ]:
"use client";

import React, { useState } from "react";
import { fetchEventSource } from "@microsoft/fetch-event-source";

export default function Stream() {
  const [messages, setMessages] = useState([]);
  const [input, setInput] = useState("");
  const [isLoading, setIsLoading] = useState(false);

  const handleSubmit = async (e) => {
    e.preventDefault();

    if (!input.trim() || isLoading) {
      return;
    }

    const currentInput = input.trim();

    // -----------------------------
    // Add user message
    // -----------------------------

    const userMessage = {
      id: crypto.randomUUID(),
      type: "user",
      content: currentInput,
    };

    // -----------------------------
    // Add temporary AI message
    // -----------------------------

    const aiMessageId = crypto.randomUUID();

    const aiMessage = {
      id: aiMessageId,
      type: "ai",
      content: "Thinking...",
    };

    setMessages((prev) => [
      ...prev,
      userMessage,
      aiMessage,
    ]);

    setInput("");
    setIsLoading(true);

    let aiResponse = "";

    const abortController = new AbortController();

    try {
      await fetchEventSource("http://localhost:8000/chat", {
        method: "POST",

        headers: {
          "Content-Type": "application/json",
          Accept: "text/event-stream",
        },

        body: JSON.stringify({
          user_input: currentInput,
        }),

        signal: abortController.signal,

        // Prevent browser caching
        cache: "no-store",

        // -----------------------------
        // Receive streaming messages
        // -----------------------------

        onmessage: (event) => {
          // -----------------------------
          // Stream completed
          // -----------------------------

          if (event.data === "[DONE]") {
            setIsLoading(false);
            abortController.abort();
            return;
          }

          try {
            const data = JSON.parse(event.data);

            // -----------------------------
            // Backend error
            // -----------------------------

            if (data.error) {
              console.error("Backend error:", data.error);

              setMessages((prev) =>
                prev.map((message) =>
                  message.id === aiMessageId
                    ? {
                        ...message,
                        content: `Error: ${data.error}`,
                      }
                    : message
                )
              );

              setIsLoading(false);
              return;
            }

            // -----------------------------
            // AI streaming chunk
            // -----------------------------

            if (data.answer) {
              aiResponse += data.answer;

              setMessages((prev) =>
                prev.map((message) =>
                  message.id === aiMessageId
                    ? {
                        ...message,
                        content: aiResponse,
                      }
                    : message
                )
              );
            }
          } catch (error) {
            console.error(
              "Error parsing SSE data:",
              error
            );
          }
        },

        // -----------------------------
        // Server closed connection
        // -----------------------------

        onclose: () => {
          setIsLoading(false);
        },

        // -----------------------------
        // Stream error
        // -----------------------------

        onerror: (error) => {
          console.error("SSE stream error:", error);

          setMessages((prev) =>
            prev.map((message) =>
              message.id === aiMessageId
                ? {
                    ...message,
                    content:
                      "Sorry, something went wrong.",
                  }
                : message
            )
          );

          setIsLoading(false);

          abortController.abort();

          throw error;
        },
      });
    } catch (error) {
      // AbortError is expected when we finish the stream
      if (error?.name !== "AbortError") {
        console.error("Fetch error:", error);
      }

      setIsLoading(false);
    }
  };

  return (
    <div
      style={{
        maxWidth: "700px",
        margin: "40px auto",
        padding: "20px",
        fontFamily: "Arial, sans-serif",
      }}
    >
      <h2>LangChain Streaming Chatbot</h2>

      {/* -------------------------------- */}
      {/* Chat messages */}
      {/* -------------------------------- */}

      <div
        style={{
          height: "400px",
          overflowY: "auto",
          border: "1px solid #ccc",
          borderRadius: "8px",
          padding: "15px",
          marginBottom: "15px",
          backgroundColor: "#fafafa",
        }}
      >
        {messages.length === 0 && (
          <div
            style={{
              color: "#888",
              textAlign: "center",
              marginTop: "150px",
            }}
          >
            Start a conversation...
          </div>
        )}

        {messages.map((message) => (
          <div
            key={message.id}
            style={{
              marginBottom: "15px",
              padding: "10px",
              borderRadius: "8px",
              backgroundColor:
                message.type === "user"
                  ? "#e8f0fe"
                  : "#eeeeee",
            }}
          >
            <strong>
              {message.type === "user"
                ? "You"
                : "AI"}
              :
            </strong>

            <div
              style={{
                marginTop: "5px",
                whiteSpace: "pre-wrap",
              }}
            >
              {message.content}
            </div>
          </div>
        ))}

        {isLoading && (
          <div
            style={{
              color: "#777",
              fontSize: "14px",
            }}
          >
            AI is typing...
          </div>
        )}
      </div>

      {/* -------------------------------- */}
      {/* Input */}
      {/* -------------------------------- */}

      <form
        onSubmit={handleSubmit}
        style={{
          display: "flex",
          gap: "10px",
        }}
      >
        <input
          type="text"
          value={input}
          onChange={(e) =>
            setInput(e.target.value)
          }
          placeholder="Type your message..."
          disabled={isLoading}
          style={{
            flex: 1,
            padding: "10px",
            border: "1px solid #ccc",
            borderRadius: "6px",
            fontSize: "16px",
          }}
        />

        <button
          type="submit"
          disabled={
            isLoading || !input.trim()
          }
          style={{
            padding: "10px 20px",
            border: "none",
            borderRadius: "6px",
            cursor:
              isLoading || !input.trim()
                ? "not-allowed"
                : "pointer",
          }}
        >
          {isLoading ? "Thinking..." : "Send"}
        </button>
      </form>
    </div>
  );
}